# Viz 1 — Descriptive Charts

Renders the descriptive-tier charts: animated scatter, variables treemap,
data-source bubble, correlation bar, cluster map, median ECI by cluster,
PCA scatter, PCA loadings, resource portfolio, rent decomposition, and
the total-vs-adjusted-rent scatter.

Reads only from `artifacts/`. No model fitting, no sample logic. If a
chart looks wrong, the fix is in the prep notebook that produced its
artifact, not here.

| Chart | Page heading | Artifact |
|---|---|---|
| 1 | Resource Production Intensity | production_per_capita_world.csv |
| 2 | Analytical Variables (Treemap) | (literal — no artifact) |
| 3 | Data Sources | (literal — no artifact) |
| 4 | Pearson Correlations with ECI | corr_with_eci.csv |
| 5 | Country Cluster Assignments | clusters_1995.csv |
| 6 | Median ECI by Cluster | median_eci_by_cluster.csv |
| 17 | Forest vs Extractive Rent Decomposition | rent_decomp_top20.csv |
| 20 | Forest-Adjusted vs Original Sample | rent_adj_vs_orig.csv |
| 25 | Resource Portfolio Composition | resource_portfolio.csv |
| 26 | PCA Scatter | pca_scatter.csv |
| 27 | PC1 Loadings | pca_loadings.csv |
| 28 | PC2 Loadings | pca_loadings.csv |

Charts 17 and 20 are descriptive in purpose (they describe the sample) and
robustness in framing (they justify a sample-selection choice). They live
here so the robustness viz notebook stays focused on regression and ML
robustness.


In [ ]:
import os, sys
sys.path.insert(0, '.')
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from _style import (base_layout, save, hex_rgba, artifact_path,
                    NAVY, GRID, BG, FONT, SUBTT)

CLUSTER_COLORS = {
    'Petrostates':        '#E63946',
    'Oil Exporters':      '#457B9D',
    'Major Producers':    '#2A9D8F',
    'Mining Exporters':   '#E9C46A',
    'Forestry Intensive': '#8B5CF6',
}

CAT_COLORS = {
    'Resource Rents':         '#E74C3C',
    'Macro & Structure':      '#8B5CF6',
    'Finance & Investment':   '#E67E22',
    'Human Capital & Infra':  '#1ABC9C',
    'Governance':             '#3498DB',
}


## Chart 1 — Resource Production Intensity (world map)

Interactive world choropleth of resource production. Two HTML
dropdowns sit above the map: **Resource** (Total / Oil / Natural
Gas / Coal / Metals) and **View** (Absolute USD / Per Capita / % of
GDP). A Plotly year slider underneath the map covers 1995–2019. Five
resources × three views = 15 traces; only one is visible at a time,
controlled by a tiny JS handler that calls `Plotly.restyle` on the
`visible` array (resource × 3 + view).

Reads the all-countries production panel (`production_panel_world.csv`),
not the 49-country forest-adjusted panel — the map shows every
country with production data, not just the regression sample. The
filename matches the page's `22_production_intensity_map.html` slot.

In [ ]:
# ── Load the production panel (all countries, 1995–2019, 5 buckets) ─────────
pp = pd.read_csv(artifact_path("production_panel_world.csv"))

BUCKETS = ["Total", "Oil", "Natural Gas", "Coal", "Metals"]
VIEWS   = [("prod_usd",     "USD",        "Absolute"),
           ("prod_pc",      "USD/person", "Per Capita"),
           ("prod_pct_gdp", "% GDP",      "% of GDP")]
YEARS   = sorted(pp["Year"].dropna().unique().astype(int).tolist())
YEARS   = [y for y in YEARS if 1995 <= y <= 2019]

YL_OR_RD = [
    [0.000, "rgb(255,255,204)"], [0.125, "rgb(255,237,160)"],
    [0.250, "rgb(254,217,118)"], [0.375, "rgb(254,178,76)"],
    [0.500, "rgb(253,141,60)"],  [0.625, "rgb(252,78,42)"],
    [0.750, "rgb(227,26,28)"],   [0.875, "rgb(189,0,38)"],
    [1.000, "rgb(128,0,38)"],
]


def _fmt_usd_compact(v):
    """Format USD as $1.2K / $1.2M / $1.2B / $1.2T."""
    if pd.isna(v): return ""
    av = abs(v)
    if av >= 1e12: return f"${v/1e12:,.2f}T"
    if av >= 1e9:  return f"${v/1e9:,.2f}B"
    if av >= 1e6:  return f"${v/1e6:,.1f}M"
    if av >= 1e3:  return f"${v/1e3:,.1f}K"
    return f"${v:,.0f}"


def _fmt_pc(v):
    if pd.isna(v): return ""
    return f"${v:,.0f}"


def _fmt_gdp_pct(v):
    if pd.isna(v): return ""
    return f"{v:.2f}%"


def _slice(year, bucket, value_col):
    sub = pp[(pp["Year"] == year) & (pp["Bucket"] == bucket)].copy()
    sub = sub.dropna(subset=[value_col, "Country Code"])
    sub = sub[sub[value_col] > 0]
    return sub.sort_values("Country Code").reset_index(drop=True)


def _customdata(values, view_key):
    fmt = {"prod_usd": _fmt_usd_compact, "prod_pc": _fmt_pc,
           "prod_pct_gdp": _fmt_gdp_pct}[view_key]
    return [fmt(v) for v in values]


# ── Build the 15 traces (default year = 2019, last year shown) ──────────────
fig00 = go.Figure()
default_year = YEARS[-1]

COLORBAR = dict(
    bgcolor="rgba(255,255,255,0.7)",
    bordercolor="#dde1e7", borderwidth=1,
    len=0.55, outlinewidth=0,
)
NAME_FOR_HOVER = {"Total": "Total", "Oil": "Oil",
                  "Natural Gas": "Natural Gas",
                  "Coal": "Coal", "Metals": "Metals"}

for ri, bucket in enumerate(BUCKETS):
    for vi, (vcol, vunit, _vlabel) in enumerate(VIEWS):
        sub = _slice(default_year, bucket, vcol)
        z = sub[vcol].astype(float).tolist()
        custom = _customdata(z, vcol)
        fig00.add_trace(go.Choropleth(
            locations=sub["Country Code"].tolist(),
            z=z,
            text=sub["Country Name"].tolist(),
            customdata=custom,
            colorscale=YL_OR_RD,
            colorbar=dict(title=dict(text=vunit,
                                     font=dict(family=FONT, size=11)),
                          **COLORBAR),
            marker=dict(line=dict(color="#c9cfd6", width=0.5)),
            hovertemplate=("<b>%{text}</b><br>"
                           f"{NAME_FOR_HOVER[bucket]}: "
                           "%{customdata}<extra></extra>"),
            visible=(ri == 0 and vi == 0),
            showscale=True,
        ))

# ── Year slider: each step restyles z / customdata / locations / text
#    for all 15 traces simultaneously ───────────────────────────────────────
steps = []
for yr in YEARS:
    z_all, cd_all, loc_all, txt_all = [], [], [], []
    for bucket in BUCKETS:
        for vcol, _vunit, _vlabel in VIEWS:
            sub = _slice(yr, bucket, vcol)
            z_all.append(sub[vcol].astype(float).tolist())
            cd_all.append(_customdata(sub[vcol].tolist(), vcol))
            loc_all.append(sub["Country Code"].tolist())
            txt_all.append(sub["Country Name"].tolist())
    steps.append(dict(
        method="restyle", label=str(yr),
        args=[dict(z=z_all, customdata=cd_all,
                   locations=loc_all, text=txt_all)],
    ))

fig00.update_geos(
    projection_type="natural earth",
    lataxis=dict(range=[-55, 85]),
    showframe=False,
    showcoastlines=True, coastlinecolor="#c9cfd6",
    showland=True, landcolor="#f0f2f5",
    showcountries=True, countrycolor="#dde1e7", countrywidth=0.5,
    bgcolor="rgba(0,0,0,0)",
)
fig00.update_layout(
    template="plotly_white",
    paper_bgcolor=BG, plot_bgcolor=BG,
    font=dict(family=FONT, size=11, color="#4b5563"),
    hoverlabel=dict(font=dict(family=FONT, size=13, color=NAVY),
                    bgcolor="white", bordercolor="#c9cfd6"),
    margin=dict(l=10, r=10, t=30, b=5),
    height=520, autosize=True,
    sliders=[dict(
        active=len(YEARS) - 1, steps=steps, len=0.92,
        x=0.04, xanchor="left", y=0, yanchor="top",
        pad=dict(b=5, t=30),
        currentvalue=dict(prefix="Year: ", visible=True,
                          xanchor="center",
                          font=dict(family=FONT, size=13, color=NAVY)),
        font=dict(family=FONT, size=11),
        transition=dict(duration=0),
    )],
)

# ── Write a custom HTML wrapper: Plotly div + two select controls + JS ─────
PLOTLY_HTML = fig00.to_html(
    full_html=True, include_plotlyjs="cdn",
    config=dict(displayModeBar=False, responsive=True),
)

CONTROLS = """
<style>body{margin:0;overflow:hidden;}</style>
<div style="position:fixed;top:10px;left:10px;z-index:1000;background:white;
padding:10px 14px;border-radius:8px;box-shadow:0 2px 10px rgba(0,0,0,0.06);
border:1px solid #dde1e7;font-family:IBM Plex Sans, -apple-system, BlinkMacSystemFont, sans-serif;">
  <label style="font-weight:600;color:#1a2744;margin-right:8px;
  font-size:13px;">Resource:</label>
  <select id="resourceSelect" style="padding:6px;border:1px solid #dde1e7;
  border-radius:4px;font-size:13px;font-family:IBM Plex Sans, -apple-system, BlinkMacSystemFont, sans-serif;">
    <option value="0">Total</option><option value="1">Oil</option>
    <option value="2">Natural Gas</option><option value="3">Coal</option>
    <option value="4">Metals</option>
  </select>
</div>
<div style="position:fixed;top:10px;right:10px;z-index:1000;background:white;
padding:10px 14px;border-radius:8px;box-shadow:0 2px 10px rgba(0,0,0,0.06);
border:1px solid #dde1e7;font-family:IBM Plex Sans, -apple-system, BlinkMacSystemFont, sans-serif;">
  <label style="font-weight:600;color:#1a2744;margin-right:8px;
  font-size:13px;">View:</label>
  <select id="normSelect" style="padding:6px;border:1px solid #dde1e7;
  border-radius:4px;font-size:13px;font-family:IBM Plex Sans, -apple-system, BlinkMacSystemFont, sans-serif;">
    <option value="0">Absolute</option><option value="1">Per Capita</option>
    <option value="2">% of GDP</option>
  </select>
</div>

<script>
let currentResource=0,currentNorm=0;
function updateMap(){
  const vis=Array(15).fill(false);
  vis[currentResource*3+currentNorm]=true;
  const p=document.getElementsByClassName('plotly-graph-div')[0];
  if(p)Plotly.restyle(p,{visible:vis});
}
setTimeout(function(){
  document.getElementById('resourceSelect').addEventListener('change',function(){
    currentResource=parseInt(this.value);updateMap();});
  document.getElementById('normSelect').addEventListener('change',function(){
    currentNorm=parseInt(this.value);updateMap();});
},100);
</script>
"""

# Inject the controls + JS just before </body>
final_html = PLOTLY_HTML.replace("</body>", CONTROLS + "\n</body>")

out_path = os.path.join(os.path.dirname(artifact_path("x")), "..", "outputs",
                        "22_production_intensity_map.html")
out_path = os.path.abspath(out_path)
os.makedirs(os.path.dirname(out_path), exist_ok=True)
with open(out_path, 'w', encoding='utf-8') as fh:
    fh.write(final_html)
n_c = pp['Country Code'].nunique()
print(f'  wrote outputs/22_production_intensity_map.html '
      f'({n_c} countries, {len(YEARS)} years, 15 traces)')

## Chart 2 (page) — Animated ECI vs log GDP per capita

This is the only chart that needs the full panel rather than a small
artifact, because the animation has one frame per year per country and
keeping that as a flat CSV would be larger than the parquet itself. We
read `panel.parquet` directly.


In [ ]:
panel = pd.read_parquet(artifact_path('panel.parquet'))
data05 = panel.dropna(subset=['Log_GDP_pc', 'Economic Complexity Index',
                              'ClusterLabels']).copy()
years = sorted(data05['Year'].unique())

cdata = {}
for cc, cdf in data05.groupby('Country Code'):
    cdf = cdf.sort_values('Year')
    if cdf['Year'].min() > 1995:
        continue
    cdata[cc] = dict(
        years = cdf['Year'].values,
        x     = cdf['Log_GDP_pc'].values,
        y     = cdf['Economic Complexity Index'].values,
        size  = cdf['Bubble_Scaled'].values,
        name  = cdf['Country Name'].iloc[0],
        clust = cdf['ClusterLabels'].iloc[0],
    )
valid_cc = list(cdata.keys())

fig01 = go.Figure()

# Initial state at year[0]: dot per country in legend-grouped traces.
for lbl in sorted(set(cdata[cc]['clust'] for cc in valid_cc)):
    color = CLUSTER_COLORS.get(lbl, '#aaa')
    cc_list = [cc for cc in valid_cc if cdata[cc]['clust'] == lbl]
    first = True
    for cc in cc_list:
        cd = cdata[cc]
        fig01.add_trace(go.Scatter(
            x=[cd['x'][0]], y=[cd['y'][0]], mode='markers+text',
            marker=dict(size=cd['size'][0], color=color, opacity=0.85,
                        line=dict(width=1.5, color='white')),
            text=[cc], textposition='top center',
            textfont=dict(size=8, color='#333'),
            name=lbl, legendgroup=lbl, showlegend=first,
            customdata=[[cd['name']]],
            hovertemplate='<b>%{customdata[0]}</b><br>'
                          'Log GDP/cap: %{x:.2f}<br>ECI: %{y:.2f}<extra></extra>',
        ))
        first = False

# One frame per year.
frames = []
for yr in years:
    fd = []
    for lbl in sorted(set(cdata[cc]['clust'] for cc in valid_cc)):
        color = CLUSTER_COLORS.get(lbl, '#aaa')
        cc_list = [cc for cc in valid_cc if cdata[cc]['clust'] == lbl]
        for cc in cc_list:
            cd = cdata[cc]
            mask = cd['years'] <= yr
            idxs = np.where(mask)[0]
            if len(idxs) == 0:
                xi, yi, si = cd['x'][0], cd['y'][0], cd['size'][0]
            else:
                i = idxs[-1]
                xi, yi, si = cd['x'][i], cd['y'][i], cd['size'][i]
            fd.append(go.Scatter(
                x=[xi], y=[yi], mode='markers+text',
                marker=dict(size=si, color=color, opacity=0.85,
                            line=dict(width=1.5, color='white')),
                text=[cc], textposition='top center',
                textfont=dict(size=8),
                customdata=[[cd['name']]],
                hovertemplate='<b>%{customdata[0]}</b><br>'
                              'Log GDP/cap: %{x:.2f}<br>ECI: %{y:.2f}<extra></extra>',
            ))
    frames.append(go.Frame(data=fd, name=str(yr)))
fig01.frames = frames

x_rng = [data05['Log_GDP_pc'].min()-0.2, data05['Log_GDP_pc'].max()+0.2]
y_rng = [data05['Economic Complexity Index'].min()-0.3,
         data05['Economic Complexity Index'].max()+0.3]

fig01.update_layout(
    **base_layout(height=620, margin=dict(l=70, r=40, t=20, b=110)),
    xaxis=dict(title='Log GDP per capita (PPP)', range=x_rng, gridcolor=GRID),
    yaxis=dict(title='Economic Complexity Index', range=y_rng, gridcolor=GRID,
               zeroline=True, zerolinecolor='#ccc'),
    legend=dict(title='Resource Profile', x=1.01, y=0.99,
                font=dict(size=11), bgcolor='rgba(250,250,250,0.9)',
                bordercolor=GRID, borderwidth=1),
    updatemenus=[dict(
        type='buttons', showactive=True, x=0.98, y=-0.13, xanchor='right',
        buttons=[
            dict(label='▶  Play', method='animate',
                 args=[None, dict(frame=dict(duration=600, redraw=True),
                                  transition=dict(duration=300))]),
            dict(label='⏸  Pause', method='animate',
                 args=[[None], dict(frame=dict(duration=0), mode='immediate')]),
        ],
    )],
    sliders=[dict(
        active=0, len=0.82, x=0.02, y=-0.10,
        currentvalue=dict(prefix='Year: ', font=dict(size=14, color=NAVY)),
        steps=[dict(
            args=[[str(y)], dict(frame=dict(duration=400, redraw=True),
                                 mode='immediate')],
            method='animate', label=str(y),
        ) for y in years],
        font=dict(size=10),
    )],
)
save(fig01, '01_eci_animated_scatter.html')


## Chart 2 — Analytical Variables (Treemap)

The variable list is a fixed taxonomy, not derived from data. It lives in
this notebook because it never changes and writing an artifact for nine
hard-coded groups would be busywork.


In [ ]:
VAR_GROUPS = {
    'Identifiers': [
        ('Country Code', 'Country Code'),
        ('Country Name', 'Country Name'),
        ('Year',         'Year'),
        ('Economic Complexity Index', 'ECI'),
    ],
    'Resource Dependence': [
        ('Oil rents (% of GDP)', 'Oil Rents'),
        ('Natural gas rents (% of GDP)', 'Gas Rents'),
        ('Mineral rents (% of GDP)', 'Mineral Rents'),
        ('Forestry rents (% of GDP)', 'Forestry Rents'),
        ('Total natural resources rents (% of GDP)', 'Total NR Rents'),
        ('Total_Production', 'Total Production'),
        ('Total_Reserves',   'Total Reserves'),
        ('Total_Production_Value', 'Production Value'),
        ('Total_Reserves_Value',   'Reserves Value'),
        ('Hydrocarbons_Dominant',     'Hydrocarbons Dom.'),
        ('Subsoil_Metals_Dominant',   'Subsoil Metals Dom.'),
        ('Precious_Metals_Dominant',  'Precious Metals Dom.'),
    ],
    'Economic Structure': [
        ('Trade (% of GDP)', 'Trade'),
        ('Industry',         'Industry'),
        ('Manufacturing',    'Manufacturing'),
        ('Agriculture',      'Agriculture'),
        ('Services',         'Services'),
    ],
    'Macroeconomic Indicators': [
        ('GDP per capita (constant prices, PPP)', 'GDP per Capita'),
        ('Share of investment in GDP',            'Investment Share'),
        ('Share of government spending in GDP',   'Gov. Spending Share'),
        ('Share of consumption in GDP',           'Consumption Share'),
        ('Inflation, consumer prices (annual %)', 'Inflation'),
        ('Gross fixed capital formation, all, Constant prices, Percent of GDP',
         'Gross Fixed Cap.'),
        ('Capital depreciation rate', 'Depreciation Rate'),
    ],
    'Fiscal & Financial': [
        ('Government revenue', 'Gov. Revenue'),
        ('Primary net lending, General government, Percent of GDP', 'Primary Net Lending'),
        ('Adjusted savings: gross savings (% of GNI)', 'Gross Savings'),
        ('Domestic credit to private sector (% of GDP)', 'Domestic Credit'),
        ('Lending interest rate (%)', 'Lending Rate'),
        ('Real interest rate (%)',    'Real Rate'),
        ('Use of IMF credit (DOD, current US$)', 'IMF Credit'),
    ],
    'Governance': [
        ('Rule of law index',          'Rule of Law'),
        ('Political corruption index', 'Pol. Corruption'),
        ('Clientelism index',          'Clientelism'),
        ('Political stability — estimate', 'Pol. Stability'),
        ('Property rights',            'Property Rights'),
    ],
    'Human Capital': [
        ('Human capital index',                 'Human Capital Index'),
        ('Death rates, crude per 1000 people',  'Death Rate'),
        ('Life expectancy at birth, total (years)', 'Life Expectancy'),
    ],
    'Infrastructure': [
        ('Access to electricity (% of population)',        'Electricity Access'),
        ('Mobile cellular subscriptions (per 100 people)', 'Mobile Subs.'),
    ],
    'Structural': [
        ('Landlocked',                                'Landlocked'),
        ('Urban population (% of total population)',  'Urban Population'),
        ('knn_reliance_pct',                          'NR Reliance (KNN)'),
    ],
}
GRP_COLORS = {
    'Identifiers': '#546E7A',  'Resource Dependence': '#E67E22',
    'Economic Structure': '#2980B9',  'Macroeconomic Indicators': '#8E44AD',
    'Fiscal & Financial': '#C0392B',  'Governance': '#16A085',
    'Human Capital': '#27AE60',       'Infrastructure': '#1A9E8F',
    'Structural': '#7D3C98',
}

t_ids, t_labels, t_parents, t_vals, t_cols, t_hover = [], [], [], [], [], []
for grp, vars_ in VAR_GROUPS.items():
    g_col = GRP_COLORS[grp]
    t_ids.append(grp); t_labels.append(grp); t_parents.append('')
    t_vals.append(len(vars_)); t_cols.append(g_col + '99')
    t_hover.append(f'{grp}<br>{len(vars_)} variables')
    for full, sh in vars_:
        t_ids.append(f'{grp}|{full}'); t_labels.append(sh); t_parents.append(grp)
        t_vals.append(1); t_cols.append(g_col); t_hover.append(full)

fig02 = go.Figure(go.Treemap(
    ids=t_ids, labels=t_labels, parents=t_parents, values=t_vals,
    customdata=t_hover, textinfo='label',
    textfont=dict(size=12, family=FONT, color='white'),
    insidetextfont=dict(size=11, family=FONT, color='white'),
    marker=dict(colors=t_cols, cornerradius=6, line=dict(width=2, color='white')),
    hovertemplate='<b>%{customdata}</b><extra></extra>',
    branchvalues='total', tiling=dict(packing='squarify', pad=3),
))
fig02.update_layout(**base_layout(height=560, margin=dict(l=10, r=10, t=20, b=10)))
save(fig02, '02_variables_treemap.html')


## Chart 3 — Data Sources Bubble Chart

Same reasoning as the treemap: hand-curated taxonomy that lives in this
notebook because it never changes.


In [ ]:
DS_MATRIX = [
    ('Resource Rents',    'World Bank',  4),
    ('Finance',           'World Bank',  4),
    ('Macro',             'World Bank',  4),
    ('GDP Structure',     'World Bank',  8),
    ('Demographics',      'World Bank',  3),
    ('Infrastructure',    'World Bank',  2),
    ('Human Capital',     'PWT 11.0',    1),
    ('Finance',           'PWT 11.0',    2),
    ('Macro',             'PWT 11.0',    2),
    ('GDP Structure',     'PWT 11.0',    3),
    ('Finance',           'IMF',         4),
    ('Macro',             'IMF',         2),
    ('Governance',        'V-Dem',      12),
    ('NR Production',     'EI / OWID',  16),
    ('NR Prices',         'EI / USGS',  16),
    ('Geography',         'Other',       1),
    ('Dependent Variable','Other',       1),
]
SRC_ORDER = ['World Bank', 'V-Dem', 'PWT 11.0', 'IMF', 'EI / OWID', 'EI / USGS', 'Other']
IND_ORDER = ['Geography','Dependent Variable','NR Prices','NR Production',
             'Resource Rents','Finance','Macro','GDP Structure',
             'Demographics','Infrastructure','Human Capital','Governance']
SRC_COLORS = {
    'World Bank':'#2980B9','V-Dem':'#E74C3C','PWT 11.0':'#27AE60',
    'IMF':'#E67E22','EI / OWID':'#8E44AD','EI / USGS':'#1ABC9C','Other':'#95A5A6',
}
ds = pd.DataFrame(DS_MATRIX, columns=['indicator','source','count'])

fig03 = go.Figure()
for src in SRC_ORDER:
    sub = ds[ds['source'] == src]
    fig03.add_trace(go.Scatter(
        x=[src]*len(sub), y=sub['indicator'], mode='markers+text',
        marker=dict(size=sub['count']*6+12, color=SRC_COLORS[src],
                    opacity=0.88, line=dict(width=1.5, color='white')),
        text=sub['count'].astype(str),
        textfont=dict(color='white', size=11, family=FONT),
        textposition='middle center', name=src,
        customdata=list(zip(sub['indicator'], sub['count'])),
        hovertemplate='<b>%{customdata[0]}</b><br>Source: '+src+
                      '<br>Variables: <b>%{customdata[1]}</b><extra></extra>',
    ))

fig03.update_layout(
    **base_layout(height=520, margin=dict(l=140, r=40, t=20, b=60)),
    xaxis=dict(categoryorder='array', categoryarray=SRC_ORDER, tickangle=-30,
               gridcolor=GRID, tickfont=dict(size=11)),
    yaxis=dict(categoryorder='array', categoryarray=IND_ORDER[::-1],
               gridcolor=GRID, tickfont=dict(size=11)),
    showlegend=False,
)
save(fig03, '03_data_sources_bubble.html')


## Chart 4 — Pearson correlations with ECI

Sorted by `r`, coloured by category. The artifact already has the right
shape so this is a pure rendering step.


In [ ]:
corr = pd.read_csv(artifact_path('corr_with_eci.csv'))

fig04 = go.Figure()
fig04.add_vline(x=0, line=dict(color='#ccc', width=1.5))
for cat in CAT_COLORS:
    sub = corr[corr['cat'] == cat]
    if len(sub) == 0:
        continue
    fig04.add_trace(go.Bar(
        y=sub['short'], x=sub['r'], orientation='h',
        name=cat, legendgroup=cat,
        marker=dict(color=CAT_COLORS[cat], opacity=0.87,
                    line=dict(color='white', width=0.4)),
        customdata=list(zip(sub['feature'], sub['r'])),
        hovertemplate='<b>%{y}</b><br>%{customdata[0]}<br>'
                      'Pearson r = %{x:.3f}<extra></extra>',
    ))
fig04.update_layout(
    **base_layout(height=600, margin=dict(l=140, r=40, t=20, b=70)),
    barmode='overlay',
    xaxis=dict(title='Pearson r with ECI', gridcolor=GRID,
               zeroline=False, range=[-0.6, 0.8]),
    yaxis=dict(gridcolor=GRID, tickfont=dict(size=11)),
    legend=dict(title='Category', x=1.01, y=0.99, font=dict(size=11),
                bgcolor='rgba(250,250,250,0.9)', bordercolor=GRID, borderwidth=1),
)
save(fig04, '04_correlation_bar.html')


## Chart 5 — Cluster world map (1995)

One choropleth trace per cluster, all with a single colour each (no scale).


In [ ]:
cl = pd.read_csv(artifact_path('clusters_1995.csv'))

fig05 = go.Figure()
for lbl in sorted(cl['ClusterLabels'].dropna().unique()):
    sub = cl[cl['ClusterLabels'] == lbl]
    color = CLUSTER_COLORS.get(lbl, '#aaa')
    fig05.add_trace(go.Choropleth(
        locations=sub['Country Code'],
        z=[1]*len(sub),
        colorscale=[[0, color], [1, color]],
        showscale=False, showlegend=True, name=lbl,
        text=sub['Country'],
        hovertemplate='<b>%{text}</b><br>'+lbl+'<extra></extra>',
        marker=dict(line=dict(color='white', width=0.7)),
    ))
fig05.update_geos(
    projection_type='natural earth',
    showcountries=True, countrycolor='#d0d0d0',
    showland=True, landcolor='#f2f4f6',
    showocean=True, oceancolor='#dce9f5',
    showframe=False,
)
fig05.update_layout(
    **base_layout(height=560, margin=dict(l=0, r=0, t=20, b=80)),
    geo=dict(bgcolor=BG),
    legend=dict(orientation='h', xanchor='center', x=0.5,
                yanchor='top', y=-0.04,
                font=dict(size=12, family=FONT),
                bgcolor='rgba(250,250,250,0.95)',
                bordercolor=GRID, borderwidth=1, tracegroupgap=0),
)
save(fig05, '05_cluster_world_map.html')


## Chart 6 — Median ECI by Cluster

Each cluster: mean line plus ±1 standard-error band.


In [ ]:
traj = pd.read_csv(artifact_path('median_eci_by_cluster.csv'))

fig06 = go.Figure()
for lbl in sorted(traj['ClusterLabels'].unique()):
    sub = traj[traj['ClusterLabels'] == lbl].sort_values('Year')
    color = CLUSTER_COLORS.get(lbl, '#999')
    fig06.add_trace(go.Scatter(
        x=sub['Year'].tolist() + sub['Year'].tolist()[::-1],
        y=(sub['median']+sub['se']).tolist() +
          (sub['median']-sub['se']).tolist()[::-1],
        fill='toself', fillcolor=hex_rgba(color, 0.12),
        line=dict(width=0), showlegend=False, hoverinfo='skip',
        legendgroup=lbl,
    ))
    fig06.add_trace(go.Scatter(
        x=sub['Year'], y=sub['median'],
        mode='lines+markers', name=lbl, legendgroup=lbl,
        line=dict(color=color, width=2.4),
        marker=dict(size=5, color=color),
        customdata=list(zip(sub['n'], sub['std'])),
        hovertemplate='<b>'+lbl+'</b> · %{x}<br>Median ECI: <b>%{y:.3f}</b>'
                      '<br>Countries: %{customdata[0]}<extra></extra>',
    ))
fig06.update_layout(
    **base_layout(height=480, margin=dict(l=70, r=40, t=20, b=60)),
    xaxis=dict(title='Year', gridcolor=GRID, dtick=5),
    yaxis=dict(title='Median ECI', gridcolor=GRID,
               zeroline=True, zerolinecolor='#ccc'),
    legend=dict(x=1.01, y=0.99, font=dict(size=11),
                bgcolor='rgba(250,250,250,0.9)',
                bordercolor=GRID, borderwidth=1),
)
save(fig06, '06_median_eci_by_cluster.html')


## Chart 17 — Forest vs extractive rent decomposition (top-20)


In [ ]:
rd = pd.read_csv(artifact_path('rent_decomp_top20.csv'))

rent_short = ['Oil', 'Gas', 'Mineral', 'Forestry']
rent_colors = {'Oil': '#1a2744', 'Gas': '#4a6fa5',
               'Mineral': '#c9a227', 'Forestry': '#2e7d4a'}

fig17 = go.Figure()
for s in rent_short:
    fig17.add_trace(go.Bar(
        y=rd['Country Name'], x=rd[s], name=s, orientation='h',
        marker_color=rent_colors[s],
        hovertemplate=f'%{{y}}<br>{s}: %{{x:.1f}}%<extra></extra>',
    ))
fig17.update_layout(
    **base_layout(height=560, margin=dict(l=160, r=40, t=20, b=60)),
    barmode='stack',
    xaxis=dict(title='Rents (% of GDP)', gridcolor=GRID),
    yaxis=dict(gridcolor=GRID, tickfont=dict(size=11)),
    legend=dict(orientation='h', x=0.5, xanchor='center', y=-0.12,
                font=dict(size=11)),
)
save(fig17, '17_forest_rent_decomposition.html')


## Chart 20 — Forest-adjusted vs original rents (2019)


In [ ]:
ra = pd.read_csv(artifact_path('rent_adj_vs_orig.csv'))
maxv = max(ra['Total_NR'].max(), ra['Adjusted_NR'].max()) * 1.05

fig20 = go.Figure()
fig20.add_trace(go.Scatter(
    x=ra['Total_NR'], y=ra['Adjusted_NR'],
    mode='markers+text', text=ra['Country Code'],
    textposition='top center', textfont=dict(size=8, color='#555'),
    marker=dict(size=8, color='#4a6fa5', opacity=0.7),
    hovertemplate='<b>%{text}</b><br>Total: %{x:.1f}%<br>'
                  'Adjusted: %{y:.1f}%<extra></extra>',
))
fig20.add_trace(go.Scatter(
    x=[0, maxv], y=[0, maxv], mode='lines',
    line=dict(color='#ccc', dash='dash', width=1),
    showlegend=False, hoverinfo='skip',
))
fig20.update_layout(
    **base_layout(height=480, margin=dict(l=70, r=40, t=20, b=60)),
    xaxis=dict(title='Total NR Rents (% of GDP)', gridcolor=GRID),
    yaxis=dict(title='Adjusted NR Rents (% of GDP)', gridcolor=GRID),
    showlegend=False,
)
save(fig20, '20_rent_adj_vs_orig_scatter.html')


## Chart 25 — Resource portfolio composition (top-20 most diversified)


In [ ]:
rp = pd.read_csv(artifact_path('resource_portfolio.csv')).iloc[::-1]

fig25 = go.Figure()
for s in ['Oil', 'Gas', 'Mineral', 'Forestry']:
    fig25.add_trace(go.Bar(
        y=rp['Country Name'], x=rp[s], name=s, orientation='h',
        marker_color={'Oil':'#1a2744','Gas':'#4a6fa5',
                       'Mineral':'#c9a227','Forestry':'#2e7d4a'}[s],
        hovertemplate=f'%{{y}}<br>{s}: %{{x:.1%}} of total<extra></extra>',
    ))
fig25.update_layout(
    **base_layout(height=580, margin=dict(l=160, r=40, t=20, b=60)),
    barmode='stack',
    xaxis=dict(title='Share of total rents', gridcolor=GRID,
               tickformat='.0%'),
    yaxis=dict(gridcolor=GRID, tickfont=dict(size=11)),
    legend=dict(orientation='h', x=0.5, xanchor='center', y=-0.10,
                font=dict(size=11)),
)
save(fig25, '25_resource_portfolio_decomposition.html')


## Chart 26 — PCA scatter (1995, coloured by cluster)


In [ ]:
ps = pd.read_csv(artifact_path('pca_scatter.csv'))
loadings = pd.read_csv(artifact_path('pca_loadings.csv'))
pc1_var = float(loadings['PC1_var'].iloc[0])
pc2_var = float(loadings['PC2_var'].iloc[0])

fig26 = go.Figure()
for lbl in sorted(ps['ClusterLabels'].dropna().unique()):
    sub = ps[ps['ClusterLabels'] == lbl]
    fig26.add_trace(go.Scatter(
        x=sub['PC1'], y=sub['PC2'], mode='markers+text',
        text=sub['Country Code'],
        textposition='top center', textfont=dict(size=8, color='#555'),
        marker=dict(size=10, color=CLUSTER_COLORS.get(lbl, '#aaa'),
                    opacity=0.85, line=dict(width=1, color='white')),
        name=lbl,
        customdata=sub[['Country', 'ClusterLabels']],
        hovertemplate='<b>%{customdata[0]}</b><br>%{customdata[1]}<br>'
                      'PC1=%{x:.2f}<br>PC2=%{y:.2f}<extra></extra>',
    ))
fig26.update_layout(
    **base_layout(height=520, margin=dict(l=70, r=40, t=20, b=60)),
    xaxis=dict(title=f'PC1 ({pc1_var*100:.1f}%)', gridcolor=GRID,
               zeroline=True, zerolinecolor='#ccc'),
    yaxis=dict(title=f'PC2 ({pc2_var*100:.1f}%)', gridcolor=GRID,
               zeroline=True, zerolinecolor='#ccc'),
    legend=dict(x=1.01, y=0.99, font=dict(size=11),
                bgcolor='rgba(250,250,250,0.9)',
                bordercolor=GRID, borderwidth=1),
)
save(fig26, '26_pca_scatter_clusters.html')


## Charts 27 and 28 — PC1 and PC2 loadings


In [ ]:
loadings = pd.read_csv(artifact_path('pca_loadings.csv'))

def loadings_chart(pc, fname):
    df = loadings.copy()
    df['abs'] = df[pc].abs()
    df = df.sort_values('abs', ascending=True)
    colors = ['#c23a3a' if v < 0 else '#1a2744' for v in df[pc]]
    fig = go.Figure(go.Bar(
        y=df['short'], x=df[pc], orientation='h',
        marker=dict(color=colors, opacity=0.9,
                    line=dict(color='white', width=0.4)),
        hovertemplate='%{y}: %{x:+.3f}<extra></extra>',
    ))
    fig.add_vline(x=0, line=dict(color='#ccc', width=1))
    fig.update_layout(
        **base_layout(height=520, margin=dict(l=180, r=40, t=20, b=60)),
        xaxis=dict(title=f'{pc} loading', gridcolor=GRID),
        yaxis=dict(gridcolor=GRID, tickfont=dict(size=11)),
        showlegend=False,
    )
    save(fig, fname)

loadings_chart('PC1', '27_pca_loadings_pc1.html')
loadings_chart('PC2', '28_pca_loadings_pc2.html')


## Done

All descriptive charts in `outputs/`. Each chart corresponds to one
artifact (or one literal taxonomy in this notebook). Re-running this
notebook is a few seconds.
